In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import matplotlib.pyplot as plt

RAW_PATH = "../data/raw/ska_songs_full.parquet"
PROCESSED_PATH = "../data/processed/ska_songs_clean.parquet"

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

In [14]:
### Load Data ###
df_raw = pd.read_parquet(RAW_PATH)
print("Rows:", len(df_raw))
print("Albums:", df_raw["album"].nunique())
df_raw.head(3)

Rows: 1174
Albums: 90


,artist,album,song_title,song_id,lyrics,year,era,city,state,label,label_type,commercial_tier
0,The Toasters,Skaboom!,Talk Is Cheap,None,"Well i heard bad news by the man in the street\nHow his shades are cool and his belts are sweet\nSka music, reggae i...",1983,foundation,New York City,New York,Moon Ska Records,ska_scene,None
1,The Toasters,Skaboom!,Pool Shark,None,"Outside in the night this is the spot\nStudio one this place is hot\nSka music, reggae, that's what we got\nAnd we'l...",1983,foundation,New York City,New York,Moon Ska Records,ska_scene,None
2,The Toasters,Skaboom!,Weekend in L.A.,None,"Arriba\n\nWell, I want to go to Rio\nSpend a week in Paris, France\nSlide off down to Haiti\nAnd do some limbo dance...",1983,foundation,New York City,New York,Moon Ska Records,ska_scene,None


In [17]:
print("Total rows:", len(df_raw))
print("Albums:", df_raw["album"].nunique())

# Check missing lyrics
print("Missing lyrics %:", df_raw["lyrics"].isna().mean())

# Check very short lyrics
df_raw["raw_len"] = df_raw["lyrics"].fillna("").str.len()
df_raw.sort_values("raw_len").head(10)[["artist","album","song_title","raw_len"]]

Total rows: 1174
Albums: 90
Missing lyrics %: 0.0


,artist,album,song_title,raw_len
879,The Chinkees,Peace Through Music,The Purpose of Education,0
880,The Chinkees,Peace Through Music,Through My Heart,0
857,The Bruce Lee Band,The Bruce Lee Band,Hongulmamotaya,0
545,Catch 22,Keasbey Nights,Riding the Fourth Wave,0
598,MU330,Chumps on Parade,Cursed Again,0
888,The Chinkees,Searching for a Brighter Future,Where’s The Avenue,0
862,The Bruce Lee Band,The Bruce Lee Band,Song 3# (Part 2),0
865,The Bruce Lee Band,The Bruce Lee Band,Komsomida,0
565,The Aquabats,The Fury of the Aquabats,Fight Song,0
868,The Chinkees,Peace Through Music,Signature,0


In [20]:
def clean_lyrics(text):
    if not isinstance(text, str):
        return ""
    
    # Remove bracketed section headers
    text = re.sub(r"\[.*?\]", "", text)

    # Remove Genius embed/footer artifacts
    text = re.sub(r"\d+Embed$", "", text)
    text = re.sub(r"Embed$", "", text)

    # Normalize the apostrophes
    text = text.replace("’", "'")

    # Remove excessive blank lines
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()

In [21]:
df_raw["lyrics_clean"] = df_raw["lyrics"].apply(clean_lyrics)

In [25]:
sample = df_raw[["lyrics", "lyrics_clean"]].sample(3, random_state=42)
sample

,lyrics,lyrics_clean
410,"Don't act up, no, don't act up, no\nEach time, I'm learning to listen\nBut what did you want to hear?\nBut what did ...","Don't act up, no, don't act up, no\nEach time, I'm learning to listen\nBut what did you want to hear?\nBut what did ..."
430,Not afraid to tell it like it is\nNot afraid to be the badass in this\nI know that things are getting better\nWantin...,Not afraid to tell it like it is\nNot afraid to be the badass in this\nI know that things are getting better\nWantin...
675,Waffle of the ancients\nWay too much of Bobby\nLaying in the gutter\nAbraham Lincoln Beard,Waffle of the ancients\nWay too much of Bobby\nLaying in the gutter\nAbraham Lincoln Beard


In [26]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)   # letters + spaces only
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_raw["lyrics_normalized"] = df_raw["lyrics_clean"].apply(normalize_text)
df_raw[["lyrics_clean", "lyrics_normalized"]].sample(3, random_state=42)

,lyrics_clean,lyrics_normalized
410,"Don't act up, no, don't act up, no\nEach time, I'm learning to listen\nBut what did you want to hear?\nBut what did ...",don t act up no don t act up no each time i m learning to listen but what did you want to hear but what did you want...
430,Not afraid to tell it like it is\nNot afraid to be the badass in this\nI know that things are getting better\nWantin...,not afraid to tell it like it is not afraid to be the badass in this i know that things are getting better wanting a...
675,Waffle of the ancients\nWay too much of Bobby\nLaying in the gutter\nAbraham Lincoln Beard,waffle of the ancients way too much of bobby laying in the gutter abraham lincoln beard


In [27]:
df_raw["word_count"] = df_raw["lyrics_normalized"].apply(lambda t: len(t.split()) if t else 0)
df_raw["char_count"] = df_raw["lyrics_clean"].str.len()

df_raw[["word_count", "char_count"]].describe()

,word_count,char_count
count,1174.000000,1174.000000
mean,234.393526,1103.518739
std,125.446307,593.039506
min,0.000000,0.000000
25%,153.000000,730.250000
50%,225.500000,1054.000000
75%,304.000000,1420.000000
max,1161.000000,5082.000000


In [28]:
df_raw.sort_values("word_count").head(20)[["artist", "album", "song_title", "word_count", "lyrics_clean"]]

,artist,album,song_title,word_count,lyrics_clean
883,The Chinkees,Peace Through Music,Will I Have a Chance?,0,
888,The Chinkees,Searching for a Brighter Future,Where’s The Avenue,0,
868,The Chinkees,Peace Through Music,Signature,0,
872,The Chinkees,Peace Through Music,Trophy Winner,0,
723,Voodoo Glow Skulls,Firme,Malas Palabras,0,
877,The Chinkees,Peace Through Music,T.J. Song,0,
266,Reel Big Fish,Why Do They Rock So Hard?,Victory Over Peter Bones (The Legend of Alan Guile Verses Peter Bones),0,
329,No Doubt,No Doubt,BND,0,
865,The Bruce Lee Band,The Bruce Lee Band,Komsomida,0,
706,Buck-O-Nine,Twenty-Eight Teeth,Peach Fish,0,


In [29]:
df_clean = df_raw[df_raw["word_count"] > 30].copy()
print("Kept:", len(df_clean), "Dropped:", len(df_raw) - len(df_clean))

Kept: 1119 Dropped: 55


In [30]:
df_clean = df_clean.drop_duplicates(subset=["artist", "album", "song_title"], keep="first")

In [31]:
### Normalize the Title ###

def normalize_title(t):
    if not isinstance(t, str):
        return ""
    t = t.lower()
    t = re.sub(r"\s*\(.*?\)\s*", "", t)   # remove parentheticals
    t = re.sub(r"\s*-\s*.*$", "", t)      # remove dash suffixes
    return t.strip()

df_clean["song_title_norm"] = df_clean["song_title"].apply(normalize_title)
df_clean = df_clean.drop_duplicates(subset=["artist","album","song_title_norm"], keep="first")

In [32]:
df_clean.columns

Index(['artist', 'album', 'song_title', 'song_id', 'lyrics', 'year', 'era',
       'city', 'state', 'label', 'label_type', 'commercial_tier', 'raw_len',
       'lyrics_clean', 'lyrics_normalized', 'word_count', 'char_count',
       'song_title_norm'],
      dtype='object')

In [33]:
(df_clean["char_count"]).describe()

count    1109.000000
mean     1160.043282
std       554.349170
min       128.000000
25%       778.000000
50%      1092.000000
75%      1438.000000
max      5082.000000
Name: char_count, dtype: float64

In [34]:
df_raw["lyrics"].str.len().describe()

count    1174.000000
mean     1106.141397
std       594.800323
min         0.000000
25%       732.250000
50%      1057.000000
75%      1424.750000
max      5095.000000
Name: lyrics, dtype: float64

In [35]:
df_clean.sort_values("word_count").head(15)[
    ["artist","album","song_title","word_count"]
]

,artist,album,song_title,word_count
968,The Impossibles,Return,Intermission,32
992,Let's Go Bowling,Mr. Twist,Cumbia del Sol,38
987,Leftöver Crack,Mediocre Generica,With the Sickness,40
639,Five Iron Frenzy,Quantity is Job 1,These Are Not My Pants (The Rock Opera) (Salsa),41
876,The Chinkees,Peace Through Music,Christmas,41
661,Five Iron Frenzy,Cheeses of Nazareth,When I See Her Face,43
53,Operation Ivy,Energy,Bankshot,47
975,Inspector 7,The Infamous,Brother vs. Brother,49
76,Skankin' Pickle,Sing Along with Skankin' Pickle,"$13,000 Is a Lot of Food",50
1090,Rancid,Life Won't Wait,Intro,56


In [36]:
album_counts = df_clean.groupby("album")["song_title"].count()
album_counts.describe()

count    90.000000
mean     12.322222
std       4.416078
min       1.000000
25%      10.250000
50%      12.000000
75%      15.000000
max      24.000000
Name: song_title, dtype: float64

In [37]:
album_counts.sort_values().head(10)

album
Searching for a Brighter Future    1
Bones                              2
Hater's Dozen                      2
Ska in Hi-Fi                       2
Give a Monkey a Brain              3
The Infamous                       3
Five to Two                        5
Fishbone                           6
Transitions                        6
Mr. Twist                          7
Name: song_title, dtype: int64

In [38]:
### Removing albums with fewer than 6 tracks ###
before = df_clean["album"].nunique()

df_clean = df_clean.groupby("album").filter(lambda x: len(x) >= 6)

after = df_clean["album"].nunique()

print("Albums before:", before)
print("Albums after:", after)
print("Albums removed:", before - after)

# I removed all of the albums where only a few songs were successfully pulled because I didn't want to distort the data

Albums before: 90
Albums after: 83
Albums removed: 7


In [ ]:
### This checks for duplicate songs ###
df_clean.duplicated(subset=["artist","album","song_title"]).sum()

np.int64(0)

In [40]:
### Checks for empty lyrics ###
(df_clean["lyrics_clean"].str.len() == 0).sum()

np.int64(0)

In [41]:
### Checks that word count distribution isn't ridiculous ###
df_clean["word_count"].describe()

count    1091.000000
mean      246.964253
std       116.814499
min        32.000000
25%       165.000000
50%       235.000000
75%       308.000000
max      1161.000000
Name: word_count, dtype: float64

In [42]:
### Checking longest songs ###
df_clean.sort_values("word_count", ascending=False).head(5)[
    ["artist", "album", "song_title", "word_count"]
]

,artist,album,song_title,word_count
435,Streetlight Manifesto,EVerything Goes Numb,Point/Counterpoint,1161
551,Catch 22,Keasbey Nights,1234 1234,952
438,Streetlight Manifesto,EVerything Goes Numb,We Are The Few,900
380,Sublime,40oz. to Freedom,Thanx,859
453,Streetlight Manifesto,Somewhere in the Between,The Receiving End of It All,789


In [43]:
### Check Album Balance ###
album_counts = df_clean.groupby("album")["song_title"].count()
album_counts.describe()

count    83.000000
mean     13.144578
std       3.499591
min       6.000000
25%      11.000000
50%      13.000000
75%      15.500000
max      24.000000
Name: song_title, dtype: float64

In [44]:
### Confirm Era and Tier Coverage ###
df_clean["era"].value_counts()
df_clean["commercial_tier"].value_counts()

Series([], Name: count, dtype: int64)

In [50]:
### Fixing missing commercial_tier ###
albums = pd.read_csv("../data/raw/albums.csv")
albums.head()

,artist,album,year,era,city,state,label,label_type,commercial_peak
0,The Toasters,Skaboom!,1983,foundation,New York City,New York,Moon Ska Records,ska_scene,1
1,The Toasters,Pool Shark,1987,foundation,New York City,New York,Moon Ska Records,ska_scene,1
2,The Toasters,New York Fever,1992,foundation,New York City,New York,Moon Ska Records,ska_scene,1
3,Fishbone,Fishbone,1985,foundation,Los Angeles,California,Columbia Records,major,2
4,Fishbone,Truth and Soul,1988,foundation,Los Angeles,California,Columbia Records,major,2


In [52]:
# 1) load albums metadata
albums.columns = albums.columns.str.strip()

# 2) rename columns
albums = albums.rename(columns={"commercial_peak": "commercial_tier"})

# 3) merge tier into dataset
df_clean = df_clean.drop(columns=["commercial_tier"], errors="ignore")

df_clean = df_clean.merge(
    albums[["artist", "album", "commercial_tier"]],
    on=["artist", "album"],
    how="left"
)

# 4) verify
print(df_clean["commercial_tier"].value_counts(dropna=False))

commercial_tier
1.0    615
2.0    360
3.0     76
NaN     25
0.0     15
Name: count, dtype: int64


In [53]:
# Finding missing commercial tier
missing = df_clean[df_clean["commercial_tier"].isna()]
missing.groupby(["artist","album"]).size().sort_values(ascending=False)

artist                        album                          
The Arrogant Sons of Bitches  Three Cheers for Dissapointment    13
Streetlight Manifesto         EVerything Goes Numb               12
dtype: int64

In [55]:
df_clean.loc[
    df_clean["album"] == "Three Cheers for Dissapointment",
    "album"
] = "Three Cheers for Disappointment"

df_clean.loc[
    df_clean["album"] == "EVerything Goes Numb",
    "album"
] = "Everything Goes Numb"

In [56]:
df_clean = df_clean.drop(columns=["commercial_tier"], errors="ignore")

df_clean = df_clean.merge(
    albums[["artist", "album", "commercial_tier"]],
    on=["artist", "album"],
    how="left"
)

In [57]:
df_clean["commercial_tier"].value_counts(dropna=False)

commercial_tier
1    628
2    372
3     76
0     15
Name: count, dtype: int64

In [62]:
print("========== DATASET SUMMARY ==========")
print("Shape:", df_clean.shape)
print("Total songs:", len(df_clean))
print("Total albums:", df_clean["album"].nunique())

print("\n========== ERA DISTRIBUTION ==========")
print(df_clean["era"].value_counts(dropna=False))

print("\n========== TIER DISTRIBUTION ==========")
print(df_clean["commercial_tier"].value_counts(dropna=False))

print("\n========== NULL CHECK ==========")
print(df_clean.isna().mean().sort_values(ascending=False).head(10))

print("\n========== DUPLICATES ==========")
print("Duplicate songs:",
      df_clean.duplicated(subset=["artist","album","song_title"]).sum())

print("\n========== ALBUM SIZE CHECK ==========")
album_counts = df_clean.groupby("album")["song_title"].count()
print("Min tracks per album:", album_counts.min())
print("Median tracks per album:", album_counts.median())
print("Max tracks per album:", album_counts.max())

print("\nSmallest albums:")
print(album_counts.sort_values().head(5))

print("\n========== RANDOM SPOT CHECK ==========")
print(df_clean.sample(5)[["artist","album","song_title","word_count"]])

========== DATASET SUMMARY ==========
Shape: (1091, 17)
Total songs: 1091
Total albums: 83

========== ERA DISTRIBUTION ==========
era
mainstream_peak    432
expansion          278
post_peak          181
foundation         103
revival             54
modern              43
Name: count, dtype: int64

========== TIER DISTRIBUTION ==========
commercial_tier
1    628
2    372
3     76
0     15
Name: count, dtype: int64

========== NULL CHECK ==========
artist               0.0
label_type           0.0
song_title_norm      0.0
char_count           0.0
word_count           0.0
lyrics_normalized    0.0
lyrics_clean         0.0
raw_len              0.0
label                0.0
album                0.0
dtype: float64

========== DUPLICATES ==========
Duplicate songs: 0

========== ALBUM SIZE CHECK ==========
Min tracks per album: 6
Median tracks per album: 13.0
Max tracks per album: 24

Smallest albums:
album
Fishbone          6
Transitions       6
Mr. Twist         7
New York Fever    8
Origin 

In [63]:
print("Any nulls left?", df_clean.isna().any().any())

Any nulls left? False


In [64]:
df_clean.to_parquet(
    "../data/processed/ska_songs_clean.parquet",
    index=False
)

print("Saved cleaned dataset.")

Saved cleaned dataset.


In [65]:
df_reload = pd.read_parquet("../data/processed/ska_songs_clean.parquet")

print("Shape:", df_reload.shape)
print("Matches in-memory:", df_reload.equals(df_clean))

Shape: (1091, 17)
Matches in-memory: True


In [66]:
print("Total songs:", len(df_clean))
print("Total albums:", df_clean["album"].nunique())
print("\nEra distribution:")
print(df_clean["era"].value_counts())
print("\nTier distribution:")
print(df_clean["commercial_tier"].value_counts())

Total songs: 1091
Total albums: 83

Era distribution:
era
mainstream_peak    432
expansion          278
post_peak          181
foundation         103
revival             54
modern              43
Name: count, dtype: int64

Tier distribution:
commercial_tier
1    628
2    372
3     76
0     15
Name: count, dtype: int64
